# 第 14 章　网络爬虫

::: {.callout-note}
## 本章要点

1. **静态页面**：`requests` + `BeautifulSoup` 爬取巨潮资讯/东方财富
2. **动态页面**：`Selenium` 处理 JavaScript 渲染
3. **数据存储**：爬取结果写入 SQLite
4. **合规说明**：`robots.txt`、请求频率、版权边界
:::


In [ ]:
# ── 第 14 章　网络爬虫　──────────────────────────────────────────────
import os, time, sqlite3, warnings
import pandas as pd
import requests
from bs4 import BeautifulSoup

warnings.filterwarnings('ignore')
OUTPUT = 'output'; os.makedirs(OUTPUT, exist_ok=True)

# 设置通用请求头（模拟浏览器，避免被直接拒绝）
HEADERS = {
    'User-Agent': ('Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
                   'AppleWebKit/537.36 (KHTML, like Gecko) '
                   'Chrome/120.0.0.0 Safari/537.36'),
    'Accept-Language': 'zh-CN,zh;q=0.9',
}
print('爬虫环境就绪 ✓')


---

## 1　合规说明

爬虫在技术上「能做」和法律上「应该做」是两回事。
在开始任何爬虫任务前，检查以下三点：

1. **`robots.txt`**：访问 `https://目标网站/robots.txt`，
   查看哪些路径被禁止爬取（`Disallow` 字段）
2. **服务条款（ToS）**：大多数数据平台明确禁止批量爬取；
   如果有官方 API（如 AKShare、CSMAR），**优先用 API**
3. **频率控制**：每次请求之间至少等待 1–2 秒，
   不要给目标服务器造成压力

::: {.callout-important}
## 学术用途与商业用途的边界

本课程爬虫内容仅用于学术研究和教学演示。
对于有明确 API 的数据源（CSMAR、AKShare、FRED），
**必须使用 API 而非爬虫**。
爬虫适合的场景：无官方 API、数据量少、有明确研究需求的公开信息。
:::


In [ ]:
# ── 2.1  静态页面：BeautifulSoup 基础 ───────────────────────────────
# 演示：解析一个模拟的 HTML 公告页面

sample_html = '''
<html><body>
<div class='announcement'>
  <h2>关于签署贷款协议的公告</h2>
  <p class='date'>公告日期：2024-03-15</p>
  <p class='content'>
    本公司于 2024 年 3 月 15 日与中国建设银行签署贷款协议，
    贷款金额为 5 亿元，贷款期限 3 年，年利率 3.85%。
  </p>
</div>
</body></html>
'''

soup = BeautifulSoup(sample_html, 'html.parser')

# 提取各字段
title   = soup.find('h2').text.strip()
date    = soup.find('p', class_='date').text.strip()
content = soup.find('p', class_='content').text.strip()

print(f'标题：{title}')
print(f'日期：{date}')
print(f'内容：{content[:60]}…')


In [ ]:
# ── 2.2  请求 + 解析模板 ────────────────────────────────────────────
def safe_get(url, headers=HEADERS, timeout=10, retry=3, sleep=1.5):
    '''带重试和延迟的安全 GET 请求。'''
    for attempt in range(retry):
        try:
            resp = requests.get(url, headers=headers, timeout=timeout)
            resp.raise_for_status()
            time.sleep(sleep + 0.5 * attempt)   # 指数退避
            return resp
        except requests.RequestException as e:
            if attempt == retry - 1:
                print(f'请求失败（{url}）：{e}')
                return None
            time.sleep(2 ** attempt)
    return None

# 测试：检查 robots.txt
print('检查东方财富 robots.txt（示例）：')
resp = safe_get('https://www.eastmoney.com/robots.txt', sleep=0.5)
if resp:
    lines = resp.text.split('\n')[:10]
    for line in lines:
        if line.strip():
            print(f'  {line.strip()}')
else:
    print('  （网络不可达，跳过）')


In [ ]:
# ── 3.1  Selenium 动态页面（代码框架）──────────────────────────────
# 演示结构，实际运行需要安装 Chrome + chromedriver

SELENIUM_CODE = '''
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time

# 无头模式（不弹出浏览器窗口）
options = webdriver.ChromeOptions()
options.add_argument('--headless')
options.add_argument('--no-sandbox')
options.add_argument('--disable-dev-shm-usage')

driver = webdriver.Chrome(options=options)

try:
    driver.get('https://目标网址')
    # 等待特定元素出现（最多 10 秒）
    elem = WebDriverWait(driver, 10).until(
        EC.presence_of_element_located((By.CLASS_NAME, '目标class')))
    content = elem.text
    print(content)
finally:
    driver.quit()    # 务必关闭，否则浏览器进程残留
'''
print('Selenium 代码框架（需安装 Chrome + chromedriver）：')
print(SELENIUM_CODE)


In [ ]:
# ── 4.1  SQLite 存储爬取结果 ─────────────────────────────────────────
DB_PATH = f'{OUTPUT}/announcements.db'

# 创建数据库和表
conn = sqlite3.connect(DB_PATH)
conn.execute('''
    CREATE TABLE IF NOT EXISTS announcements (
        id          INTEGER PRIMARY KEY AUTOINCREMENT,
        stkcd       TEXT,
        ann_date    TEXT,
        title       TEXT,
        url         TEXT,
        content     TEXT,
        created_at  DATETIME DEFAULT CURRENT_TIMESTAMP
    )
''')
conn.commit()

# 写入模拟数据
sample_records = [
    ('000001', '2024-03-15', '关于签署贷款协议的公告',
     'https://example.com/ann/1', '与建设银行签署 5 亿元贷款协议…'),
    ('000002', '2024-03-16', '关于重大资产重组的提示性公告',
     'https://example.com/ann/2', '公司正在筹划重大资产重组…'),
]
conn.executemany(
    'INSERT INTO announcements (stkcd,ann_date,title,url,content) VALUES (?,?,?,?,?)',
    sample_records
)
conn.commit()

# 读取验证
df_db = pd.read_sql('SELECT * FROM announcements', conn)
conn.close()
print(f'SQLite 数据库：{DB_PATH}')
print(df_db[['stkcd','ann_date','title']].to_string(index=False))


---

## 5　章末练习

**练习 1（robots.txt 检查）**
分别检查巨潮资讯（cninfo.com.cn）和东方财富（eastmoney.com）的 robots.txt，
列出哪些路径被禁止爬取，讨论这对学术研究数据获取的影响。

**练习 2（BeautifulSoup 解析）**
写一个函数，输入 HTML 字符串，
用正则表达式从公告内容中提取「贷款金额」「利率」「期限」三个字段，
输出结构化字典。

**练习 3（SQLite 查询）**
在 announcements 数据库中，
写 SQL 查询按股票代码统计公告数量，
用 `pandas.read_sql` 读取结果并绘制条形图。
